# Extract, Transform, Load (ETL)
## Paycheck Protection Program (PPP) Loan Fraud Analysis

## Goal
The goal of this notebook is to clean and prepare the PPP dataset for SQL analysis.
All issues identified in the EDA phase are resolved here.

Input: data/raw/public_150k_plus_240930.csv
Output: data/cleaned/cleaned_ppp.csv

## Steps:
1. Drop columns defined in EDA 
2. Convert date columns from string to datetime format
3. Investigate and handle $0 loan amounts
4. Fill missing values in BusinessType, NAICSCode, BusinessAgeDescription
5. Check for duplicates
6. Export cleaned dataset

In [19]:
import pandas as pd 

RAW_FILE = '../data/raw/public_150k_plus_240930.csv'
df = pd.read_csv(RAW_FILE, encoding='latin-1')

## 0. Datashape before cleaning

In [20]:
print(f'Cleaned dataset:  {df.shape[0]:,} rows, {df.shape[1]} columns')

Cleaned dataset:  968,524 rows, 53 columns


## 1. Dropping columns defined in EDA
30 columns identified in EDA as not relevant for fraud analysis are dropped here.

Dropped categories:
- SBA internal codes (2)
- Servicing lender address details (5)
- Originating lender internal ID (1)
- Redundant location data (4)
- Demographic data (4)
- Location indicators (3)
- Not needed (3)
- Proceed fields with high missing rates (8)

In [21]:
df = df.drop(columns=[
        # SBA internal codes
    'SBAOfficeCode', 'SBAGuarantyPercentage',
    # Servicing lender address details
    'ServicingLenderLocationID', 'ServicingLenderAddress', 'ServicingLenderCity',
    'ServicingLenderState', 'ServicingLenderZip',
    # Originating lender internal ID
    'OriginatingLenderLocationID',
    # Redundant location data
    'ProjectCity', 'ProjectCountyName', 'ProjectZip', 'CD',
    # Demographic data
    'Race', 'Ethnicity', 'Gender', 'Veteran',
    # Location indicators
    'RuralUrbanIndicator', 'HubzoneIndicator', 'LMIIndicator',
    # Not needed
    'LoanNumber', 'Term', 'UndisbursedAmount',
    # Proceed fields
    'FranchiseName', 'NonProfit', 'UTILITIES_PROCEED', 'MORTGAGE_INTEREST_PROCEED',
    'RENT_PROCEED', 'REFINANCE_EIDL_PROCEED', 'HEALTH_CARE_PROCEED', 'DEBT_INTEREST_PROCEED'
], errors='ignore')

print(f'Columns after dropping: {df.shape[1]}')

Columns after dropping: 23


## 2. Conveting date values to date data type 
Columns DateApproved, LoanStatusDate, ForgivenessDate are orginally in string data type.
For better handing with dates, those columns need to be transformed to datetime data type.

In [22]:
df['DateApproved'] = pd.to_datetime(df['DateApproved'])
df['LoanStatusDate'] = pd.to_datetime(df['LoanStatusDate'])
df['ForgivenessDate'] = pd.to_datetime(df['ForgivenessDate'])

print(df[['DateApproved', 'LoanStatusDate', 'ForgivenessDate']].dtypes)

DateApproved       datetime64[ns]
LoanStatusDate     datetime64[ns]
ForgivenessDate    datetime64[ns]
dtype: object


## 3. Loans with $0 values

In [23]:
zero_loans = df[df['InitialApprovalAmount'] == 0]
print(f'Loans with $0 amount: {len(zero_loans)}')
display(df[df['InitialApprovalAmount'] == 0][['BorrowerName', 'InitialApprovalAmount', 'CurrentApprovalAmount', 'ForgivenessAmount', 'LoanStatus', 'JobsReported']])

df.loc[df['InitialApprovalAmount'] == 0, 'InitialApprovalAmount'] = df['CurrentApprovalAmount']
print(f'Zero loans after fix: {(df["InitialApprovalAmount"] == 0).sum()}')

Loans with $0 amount: 2


,BorrowerName,InitialApprovalAmount,CurrentApprovalAmount,ForgivenessAmount,LoanStatus,JobsReported
575278,MELTON SALES AND SERVICE INC,0.0,331000.0,335361.95,Paid in Full,28.0
892298,"INDUSTRIAL HORSEPOWER PLUS, INC.",0.0,250800.0,254242.49,Paid in Full,15.0


Zero loans after fix: 0


### Finding:
All $0 InitialApprovalAmount values replaced with CurrentApprovalAmount.
These appear to be data entry errors since CurrentApprovalAmount shows valid amounts.

## 4. Handling a null values

In [24]:
df['BusinessType'] = df['BusinessType'].fillna('Unknown')
df['NAICSCode'] = df['NAICSCode'].fillna('Unknown')
df['BusinessAgeDescription'] = df['BusinessAgeDescription'].fillna('Unknown')

### Finding:
BusinessType, NAICSCode and BusinessAgeDescription null values filled with 'Unknown'.

## 5. Check for duplicates

In [25]:
print(f'Duplicate rows: {df.duplicated().sum()}')

Duplicate rows: 0


### Finding:
No duplicate rows found, dataset is clean in this regard

## Export cleaned dataset

In [26]:
print(f'Original dataset: 968,524 rows, 53 columns')
print(f'Cleaned dataset:  {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Columns removed:  {53 - df.shape[1]}')
df.to_csv('../data/cleaned/cleaned_ppp.csv', index=False)
print(f'Exported: {df.shape[0]:,} rows, {df.shape[1]} columns')

Original dataset: 968,524 rows, 53 columns
Cleaned dataset:  968,524 rows, 23 columns
Columns removed:  30
Exported: 968,524 rows, 23 columns


## ETL Summary

### Changes Made
1. Dropped 30 columns not relevant for fraud analysis
2. Converted DateApproved, LoanStatusDate and ForgivenessDate to datetime format
3. Fixed 2 loans with $0 InitialApprovalAmount, replaced with CurrentApprovalAmount
4. Filled null values in BusinessType, NAICSCode and BusinessAgeDescription with 'Unknown'
5. No duplicate rows found

### Dataset Before and After
- Before, 968,524 rows, 53 columns
- After, 968,524 rows, 23 columns
- Columns removed, 30

### Next Step
Load cleaned_ppp.csv into PostgreSQL for fraud analysis.